# 04 - 用预训练模型推理**学习目标：**- 掌握 YOLO 推理的完整流程：前处理 → 推理 → 后处理- 理解 NMS（非极大值抑制）的工作原理- 学会调整置信度阈值和 IoU 阈值- 对比不同模型大小的效果---

## 1. 推理流程概览```原始图片 (任意尺寸)    │    ▼┌──────────┐│  前处理   │  Resize + Letterbox + Normalize + HWC→CHW└────┬─────┘     │  Tensor (1, 3, 640, 640)     ▼┌──────────┐│  模型推理 │  Backbone → Neck → Head└────┬─────┘     │  Raw predictions (1, 84, 8400)  ← 8400 个候选框     ▼┌──────────┐│  后处理   │  置信度过滤 + NMS└────┬─────┘     │  Final detections     ▼  可视化输出```

## 2. 加载模型并推理

In [ ]:
from ultralytics import YOLOfrom PIL import Imageimport requestsfrom io import BytesIO# 加载预训练模型model = YOLO("yolo11n.pt")# 下载示例图片url = "https://ultralytics.com/images/bus.jpg"img = Image.open(BytesIO(requests.get(url).content))print(f"图片尺寸: {img.size}")

In [ ]:
# 推理 - 就这么简单！results = model(img, verbose=False)# 查看结果结构result = results[0]print(f"检测到 {len(result.boxes)} 个目标\n")for box in result.boxes:    cls_id = int(box.cls[0])    conf = float(box.conf[0])    xyxy = box.xyxy[0].tolist()    print(f"  {model.names[cls_id]:<15} {conf:.2%}  [{xyxy[0]:.0f},{xyxy[1]:.0f},{xyxy[2]:.0f},{xyxy[3]:.0f}]")

In [ ]:
# 可视化检测结果annotated = result.plot()  # numpy array (BGR)from PIL import ImageImage.fromarray(annotated[..., ::-1])  # BGR → RGB

## 3. NMS 详解**NMS = Non-Maximum Suppression（非极大值抑制）**为什么需要 NMS？因为模型会对同一个目标产生多个重叠的预测框：```NMS 之前：                    NMS 之后：┌──────────────┐             ┌──────────────┐│ ┌──────────┐ │             │              ││ │ ┌──────┐ │ │             │   🚗 0.95    ││ │ │🚗0.95│ │ │     →       │              ││ │ └──────┘ │ │             └──────────────┘│ │  🚗0.82  │ ││ └──────────┘ ││    🚗0.71    │└──────────────┘  3 个重叠框          只保留最自信的 1 个```**NMS 算法步骤**：1. 按置信度降序排列所有框2. 取最高分的框加入结果集3. 计算它与剩余所有框的 IoU4. 删除 IoU > 阈值的框（它们检测的是同一目标）5. 重复直到没有框剩余

In [ ]:
# 教学演示 — 下面的实现帮助理解原理，生产代码请用 yolo_learn.*
# 手动实现 NMS（帮助理解原理）import numpy as npdef compute_iou(box1, box2):    """计算两个框的 IoU"""    x1 = max(box1[0], box2[0])    y1 = max(box1[1], box2[1])    x2 = min(box1[2], box2[2])    y2 = min(box1[3], box2[3])    intersection = max(0, x2-x1) * max(0, y2-y1)    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])    union = area1 + area2 - intersection    return intersection / union if union > 0 else 0def my_nms(boxes, scores, iou_threshold=0.5):    """手动 NMS 实现"""    order = scores.argsort()[::-1]  # 按分数降序    keep = []    while len(order) > 0:        i = order[0]        keep.append(i)        if len(order) == 1:            break        remaining = order[1:]        ious = [compute_iou(boxes[i], boxes[j]) for j in remaining]        mask = [iou < iou_threshold for iou in ious]        order = remaining[mask]    return keep# 测试print("NMS 测试：")boxes = np.array([[100, 100, 200, 200], [105, 105, 205, 205], [300, 300, 400, 400]])scores = np.array([0.9, 0.85, 0.7])kept = my_nms(boxes, scores, iou_threshold=0.5)print(f"  输入 {len(boxes)} 个框, 保留 {len(kept)} 个: {kept}")print(f"  框0和框1的 IoU = {compute_iou(boxes[0], boxes[1]):.3f}")

## 4. 阈值参数的影响

In [ ]:
# 置信度阈值 vs 检测数量print("置信度阈值对检测结果的影响：")print(f"{'阈值':<8} {'检测数':<8} {'结果'}")print("-" * 40)for conf in [0.1, 0.25, 0.5, 0.75, 0.9]:    r = model(img, conf=conf, verbose=False)    names = [model.names[int(b.cls[0])] for b in r[0].boxes]    print(f"{conf:<8} {len(r[0].boxes):<8} {names}")

In [ ]:
# IoU 阈值对 NMS 的影响print("\nIoU 阈值对 NMS 的影响：")print(f"{'IoU阈值':<10} {'检测数':<8}")print("-" * 20)for iou in [0.3, 0.5, 0.7, 0.9]:    r = model(img, conf=0.25, iou=iou, verbose=False)    print(f"{iou:<10} {len(r[0].boxes):<8}")

## 5. 批量推理可以一次处理多张图片：

In [ ]:
from pathlib import Path# 从 COCO128 中取几张图片img_dir = Path("../data/coco128/images/train2017")imgs = sorted(img_dir.glob("*.jpg"))[:5]# 批量推理results = model([str(p) for p in imgs], verbose=False)print("批量推理结果：")for img_path, r in zip(imgs, results):    n = len(r.boxes)    classes = set(model.names[int(b.cls[0])] for b in r.boxes)    print(f"  {img_path.name}: {n} 个目标 → {classes}")

## 6. 不同模型大小对比

In [ ]:
import time

test_img = str(imgs[0])
print(f"对比不同模型在 {Path(test_img).name} 上的表现：\n")
print(f"{'模型':<12} {'参数量':<12} {'检测数':<8} {'耗时':<10}")
print("-" * 45)

for size in ['n', 's', 'm']:
    m = YOLO(f"yolo11{size}.pt")
    params = sum(p.numel() for p in m.model.parameters())
    
    # 预热
    m(test_img, verbose=False)
    
    # 计时
    start = time.time()
    for _ in range(3):
        r = m(test_img, verbose=False)
    elapsed = (time.time() - start) / 3
    
    print(f"yolo11{size}    {params:>10,}    {len(r[0].boxes):<8} {elapsed*1000:.0f}ms")

## 7. 核心概念总结| 概念 | 说明 ||------|------|| **前处理** | Resize、归一化、通道转换 (HWC→CHW) || **置信度** | 模型对检测结果的确信程度 (0-1) || **NMS** | 去除重复检测框的算法 || **IoU 阈值** | NMS 中判断"重复"的标准，越大越严格 || **置信度阈值** | 过滤低置信度检测的阈值 |---**上一节**: [03 - YOLO 架构](03_yolo_architecture.ipynb)**下一节**: [05 - 训练基础](05_training_basics.ipynb) - 理解训练过程